# Chatbot using Hugging Face Transformers
### Data Science Internship – February 2026 | Assignment NLP-3

**Objective:** Build a conversational chatbot using a pre-trained transformer model from Hugging Face that dynamically generates meaningful responses.

---

**Pipeline Flow:**
```
User Input → Model Processing → Response Generation → Display Output → Loop Until Exit
```

**Model Used:** `microsoft/DialoGPT-medium` — a pre-trained conversational model fine-tuned on dialogue data.

---
## Step 0: Install Required Libraries

In [ ]:
# Install Hugging Face Transformers and PyTorch
# Uncomment and run this cell if you are on Google Colab
# !pip install transformers torch --quiet

---
## Step 1: Import Libraries

In [ ]:
# Core libraries
import torch
import warnings
warnings.filterwarnings('ignore')

# Hugging Face Transformers
from transformers import AutoModelForCausalLM, AutoTokenizer

print('All libraries imported successfully.')
print(f'PyTorch version : {torch.__version__}')
print(f'Device          : {"GPU (CUDA)" if torch.cuda.is_available() else "CPU"}')

---
## Step 2: Load Pre-trained Transformer Model

We use **DialoGPT-medium** from Microsoft, available on the Hugging Face Model Hub.

DialoGPT is a large-scale pre-trained dialogue response generation model trained on 147M Reddit comment chains. It is specifically fine-tuned for multi-turn conversations, making it ideal for a chatbot application.

- **Model card:** https://huggingface.co/microsoft/DialoGPT-medium
- **Architecture:** GPT-2 based causal language model
- **Training data:** 147M Reddit conversations

In [ ]:
# Define the model name from Hugging Face Hub
MODEL_NAME = 'microsoft/DialoGPT-medium'

print(f'Loading tokenizer from: {MODEL_NAME}')
print('This may take a moment on first run as the model downloads...')
print()

# Load tokenizer
# The tokenizer converts raw text into token IDs the model understands
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token   # set pad token to EOS

# Load the pre-trained causal language model
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model.eval()   # set to evaluation mode (no gradient updates)

# Use GPU if available, otherwise CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = model.to(device)

print(f'Model loaded successfully.')
print(f'Model      : {MODEL_NAME}')
print(f'Device     : {device}')
print(f'Parameters : {sum(p.numel() for p in model.parameters()):,}')

---
## Step 3: Build the Response Generation Function

This function takes the full conversation history and generates the next response. Maintaining conversation history allows the model to produce contextually relevant replies across multiple turns.

In [ ]:
def generate_response(user_input, conversation_history, max_history_turns=5):
    """
    Generate a chatbot response using DialoGPT.

    Args:
        user_input          (str)  : The latest message from the user.
        conversation_history (list): List of past token ID tensors.
        max_history_turns    (int) : Max number of past turns to keep in context.

    Returns:
        response_text (str)  : The generated response string.
        updated_history (list): Updated conversation history.
    """

    # --- Tokenize the new user input ---
    # Append EOS token so the model knows where the user turn ends
    new_input_ids = tokenizer.encode(
        user_input + tokenizer.eos_token,
        return_tensors='pt'
    ).to(device)

    # --- Build conversation context ---
    # Keep only the last N turns to avoid exceeding the model's max token limit
    recent_history = conversation_history[-max_history_turns:]

    if recent_history:
        # Concatenate past conversation with new input
        bot_input_ids = torch.cat(recent_history + [new_input_ids], dim=-1)
    else:
        # First turn: no history yet
        bot_input_ids = new_input_ids

    # --- Ensure we stay within model's max token length (1024 for DialoGPT) ---
    if bot_input_ids.shape[-1] > 900:
        # Trim from the front, keeping the most recent context
        bot_input_ids = bot_input_ids[:, -900:]

    # --- Generate response ---
    with torch.no_grad():
        output_ids = model.generate(
            bot_input_ids,
            max_new_tokens      = 150,       # max tokens in the response
            pad_token_id        = tokenizer.eos_token_id,
            do_sample           = True,      # enable sampling for varied responses
            top_k               = 50,        # top-k sampling: consider top 50 tokens
            top_p               = 0.92,      # nucleus sampling: cumulative probability
            temperature         = 0.75,      # controls randomness (lower = more focused)
            repetition_penalty  = 1.3,       # penalise repeating the same phrases
            no_repeat_ngram_size= 3,         # prevent repeating 3-grams
        )

    # --- Decode only the new tokens (skip the input tokens) ---
    response_ids  = output_ids[:, bot_input_ids.shape[-1]:]
    response_text = tokenizer.decode(response_ids[0], skip_special_tokens=True).strip()

    # --- Fallback if model returns empty response ---
    if not response_text:
        response_text = "I am not sure I understood that. Could you rephrase?"

    # --- Update conversation history with the full exchange ---
    updated_history = recent_history + [output_ids]

    return response_text, updated_history


print('Response generation function defined successfully.')

---
## Step 4: Build the Full Chatbot Interface

The chatbot accepts user input, generates a response, and loops continuously until the user types `exit` or `quit`.

In [ ]:
def run_chatbot():
    """
    Main chatbot loop.

    - Greets the user on startup.
    - Accepts multi-turn conversation input.
    - Maintains conversation history for context-aware responses.
    - Exits cleanly when user types 'exit' or 'quit'.
    """

    print('=' * 60)
    print('         AI CHATBOT  |  Powered by DialoGPT')
    print('=' * 60)
    print('Chatbot: Hello! I am your AI assistant.')
    print('         How can I help you today?')
    print('         (Type "exit" or "quit" to end the conversation)')
    print('-' * 60)

    # Stores the token tensors of past conversation turns
    conversation_history = []

    while True:
        # --- Get user input ---
        try:
            user_input = input('You: ').strip()
        except (EOFError, KeyboardInterrupt):
            # Handle Ctrl+C or end of input stream gracefully
            print('\nChatbot: Goodbye! Have a great day.')
            break

        # --- Validate input ---
        if not user_input:
            print('Chatbot: Please type something so I can respond.')
            continue

        # --- Exit condition ---
        if user_input.lower() in ['exit', 'quit', 'bye', 'goodbye']:
            print('Chatbot: It was great talking to you. Goodbye!')
            print('=' * 60)
            break

        # --- Generate response ---
        try:
            response, conversation_history = generate_response(
                user_input,
                conversation_history
            )
            print(f'Chatbot: {response}')
            print('-' * 60)

        except Exception as e:
            # Graceful error handling — chatbot does not crash
            print(f'Chatbot: I encountered an issue generating a response. Please try again.')
            print(f'         (Error: {e})')
            conversation_history = []   # reset history on error


# Run the chatbot
run_chatbot()

---
## Step 5: Simulated Conversation Output

The cell below demonstrates a pre-recorded chatbot session for notebook submission purposes. This shows the expected conversation flow without requiring live user input.

In [ ]:
def demo_conversation():
    """
    Runs a predefined list of user inputs through the chatbot
    and prints the full conversation. Used for demonstration
    and submission output purposes.
    """

    # Predefined user inputs for demonstration
    demo_inputs = [
        "Hello",
        "What is Artificial Intelligence?",
        "Who created Python?",
        "What is machine learning?",
        "Can you tell me about natural language processing?",
        "What is a transformer model?",
        "Thank you",
        "exit"
    ]

    print('=' * 60)
    print('    DEMO CONVERSATION  |  AI Chatbot using DialoGPT')
    print('=' * 60)
    print('Chatbot: Hello! I am your AI assistant.')
    print('         How can I help you today?')
    print('-' * 60)

    conversation_history = []

    for user_input in demo_inputs:
        print(f'You    : {user_input}')

        # Exit condition
        if user_input.lower() in ['exit', 'quit', 'bye', 'goodbye']:
            print('Chatbot: It was great talking to you. Goodbye!')
            print('=' * 60)
            break

        # Generate response
        try:
            response, conversation_history = generate_response(
                user_input,
                conversation_history
            )
            print(f'Chatbot: {response}')
        except Exception as e:
            print(f'Chatbot: Sorry, I had trouble generating a response. ({e})')
            conversation_history = []

        print('-' * 60)


# Run the demonstration
demo_conversation()

---
## Step 6: Understanding the Model — How DialoGPT Works

### Architecture

DialoGPT is built on the GPT-2 architecture, which is a **causal language model** (also called an autoregressive model). This means it predicts the next token based on all the tokens that came before it — one token at a time, from left to right.

### How text generation works

When you pass a conversation history to DialoGPT, it:
1. Tokenizes all the text into a sequence of integer IDs
2. Passes the sequence through multiple transformer layers, each applying self-attention
3. Produces a probability distribution over the entire vocabulary for the next token
4. Samples from that distribution according to the generation strategy
5. Appends the chosen token and repeats until it produces an EOS token

### Generation parameters explained

| Parameter | Value | Effect |
|---|---|---|
| `temperature` | 0.75 | Lower values make responses more focused and predictable |
| `top_k` | 50 | Only consider the 50 most likely next tokens at each step |
| `top_p` | 0.92 | Nucleus sampling — use smallest set of tokens covering 92% probability |
| `repetition_penalty` | 1.3 | Penalise tokens that have already appeared to avoid loops |
| `no_repeat_ngram_size` | 3 | Prevent any 3-word sequence from being repeated |

### Why conversation history matters

Without history, every response is generated from scratch with no context. By appending previous turns to the input, the model can produce responses that are coherent with what was said earlier in the conversation. This is what makes it feel like a real multi-turn dialogue rather than a series of disconnected one-shot responses.

---
## Step 7: Summary & Findings

| Component | Detail |
|---|---|
| Model | microsoft/DialoGPT-medium |
| Architecture | GPT-2 based causal language model |
| Parameters | ~345 million |
| Tokenizer | Byte-Pair Encoding (BPE) |
| Generation Strategy | Top-k + Nucleus (top-p) sampling |
| Context Window | Up to 1024 tokens |
| Multi-turn Support | Yes — via conversation history |
| Exit Handling | 'exit' / 'quit' / 'bye' / 'goodbye' |
| Error Handling | Graceful fallback with history reset |

### Key observations

1. **Conversation history is essential.** Without it, the model has no context and produces generic responses. Maintaining history across turns significantly improves coherence.

2. **Temperature controls creativity.** A temperature of 0.75 strikes a balance between focused and varied responses. Lower values make the chatbot more repetitive; higher values make it more random.

3. **History length needs a cap.** DialoGPT has a maximum input length of 1024 tokens. Without trimming, long conversations exceed this limit and cause errors. Keeping only the most recent turns avoids this.

4. **Repetition penalty is important.** Without it, the model tends to repeat phrases from earlier in the conversation. A penalty of 1.3 keeps responses fresh.

5. **DialoGPT vs GPT-2.** While both share the same architecture, DialoGPT is specifically fine-tuned on dialogue data, which means its responses are more conversational in tone compared to raw GPT-2.